# IMDB Data Analysis - Final Big Data Project

**Group Members:**
- [Student Name 1]
- [Student Name 2]
- [Student Name 3]
- [Student Name 4]

## Project Description
This project performs comprehensive analysis of IMDB movie data and implements a Kafka-based stream processing system to monitor Wikimedia events in real-time.

## Install Dependencies

Run this cell once to install required packages.

In [1]:
!pip install pandas requests kafka-python sseclient-py


   -------------------- ------------------- 1/2 [kafka-python]
   -------------------- ------------------- 1/2 [kafka-python]
   -------------------- ------------------- 1/2 [kafka-python]
   -------------------- ------------------- 1/2 [kafka-python]
   ---------------------------------------- 2/2 [kafka-python]



## Imports and Configuration

In [2]:
import pandas as pd
import requests
import os
from datetime import datetime
from pathlib import Path

# Configuration
DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)

BASE_URL = 'https://datasets.imdbws.com/'

print("✓ Configuration complete")
print(f"Data directory: {DATA_DIR.absolute()}")

✓ Configuration complete
Data directory: c:\Users\gblia\OneDrive\Bureau\Projet_BigData\data


---
# Part 1: IMDB Data Analysis
---

## Question 1: Load Data from IMDB

Automated download and loading of IMDB datasets.

In [3]:
# List of datasets to download
datasets = [
    'name.basics.tsv.gz',
    'title.akas.tsv.gz',
    'title.basics.tsv.gz',
    'title.crew.tsv.gz',
    'title.episode.tsv.gz',
    'title.principals.tsv.gz',
    'title.ratings.tsv.gz'
]

def download_dataset(filename):
    """
    Downloads an IMDB dataset file if it doesn't exist.
    
    Args:
        filename: Name of the file to download
    Returns:
        Path to the downloaded file
    """
    url = BASE_URL + filename
    filepath = DATA_DIR / filename
    
    if filepath.exists():
        print(f"✓ {filename} already exists")
        return str(filepath)
    
    print(f"Downloading {filename}...")
    response = requests.get(url, stream=True)
    
    if response.status_code == 200:
        total_size = int(response.headers.get('content-length', 0))
        with open(filepath, 'wb') as f:
            downloaded = 0
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
                downloaded += len(chunk)
                if total_size > 0:
                    percent = (downloaded / total_size) * 100
                    print(f"\r  Progress: {percent:.1f}%", end='')
        print(f"\n✓ {filename} downloaded successfully")
    else:
        print(f"✗ Error downloading {filename}: {response.status_code}")
    
    return str(filepath)

# Download all datasets
print("=" * 60)
print("DOWNLOADING IMDB DATASETS")
print("=" * 60)
print("Note: This may take 10-20 minutes depending on your connection\n")

for dataset in datasets:
    download_dataset(dataset)

print("\n" + "=" * 60)
print("✓ All datasets downloaded")
print("=" * 60)

DOWNLOADING IMDB DATASETS
Note: This may take 10-20 minutes depending on your connection

  Progress: 100.0%
✓ name.basics.tsv.gz downloaded successfully
  Progress: 100.0%
✓ title.akas.tsv.gz downloaded successfully
  Progress: 100.0%
✓ title.basics.tsv.gz downloaded successfully
  Progress: 100.0%
✓ title.crew.tsv.gz downloaded successfully
  Progress: 100.0%
✓ title.episode.tsv.gz downloaded successfully
  Progress: 100.0%
✓ title.principals.tsv.gz downloaded successfully
  Progress: 100.0%
✓ title.ratings.tsv.gz downloaded successfully

✓ All datasets downloaded


In [5]:
def load_imdb_data(filename):
    """
    Loads a compressed IMDB TSV file.
    Replaces '\\N' with NaN for missing values.
    
    Args:
        filename: Name of file to load
    Returns:
        DataFrame with the loaded data
    """
    filepath = DATA_DIR / filename
    print(f"Loading {filename}...", end=' ')
    
    df = pd.read_csv(
        filepath,
        sep='\t',
        compression='gzip',
        na_values='\\N',
        low_memory=False
    )
    
    print(f"✓ {len(df):,} rows, {len(df.columns)} columns")
    return df

# Load primary datasets
print("\nLOADING DATASETS INTO MEMORY")
print("=" * 60)

name_basics = load_imdb_data('name.basics.tsv.gz')
title_basics = load_imdb_data('title.basics.tsv.gz')
title_crew = load_imdb_data('title.crew.tsv.gz')
title_ratings = load_imdb_data('title.ratings.tsv.gz')
title_akas = load_imdb_data('title.akas.tsv.gz')

print("\n✓ All datasets loaded successfully")


LOADING DATASETS INTO MEMORY
Loading name.basics.tsv.gz... ✓ 14,882,054 rows, 6 columns
Loading title.basics.tsv.gz... 

MemoryError: Unable to allocate 11.5 MiB for an array with shape (12068990,) and data type bool

In [ ]:
# Preview the data
print("\nDATA PREVIEW")
print("=" * 60)
print("\nname_basics (People):")
display(name_basics.head())
print(f"\nColumns: {list(name_basics.columns)}")

print("\n" + "-" * 60)
print("\ntitle_basics (Titles):")
display(title_basics.head())
print(f"\nColumns: {list(title_basics.columns)}")

## Question 2: Total Number of People

Calculate total number of individuals in the dataset.

In [ ]:
total_people = len(name_basics)
print(f"\nTotal number of people in dataset: {total_people:,}")

**ANSWER:** The total number of people in the dataset is **[TO BE FILLED AFTER EXECUTION]**

## Question 3: Earliest Year of Birth

In [ ]:
# Convert birthYear to numeric
name_basics['birthYear'] = pd.to_numeric(name_basics['birthYear'], errors='coerce')

# Find earliest birth year
earliest_birth_year = name_basics['birthYear'].min()
print(f"Earliest year of birth: {int(earliest_birth_year)}")

# Find person(s) with this birth year
oldest_person = name_basics[name_basics['birthYear'] == earliest_birth_year]
print(f"\nPerson(s) born in {int(earliest_birth_year)}:")
display(oldest_person[['nconst', 'primaryName', 'birthYear', 'deathYear', 'primaryProfession']])

**ANSWER:** The earliest year of birth is **[TO BE FILLED]**

## Question 4: Years Since Birth

In [ ]:
current_year = datetime.now().year
years_ago = current_year - int(earliest_birth_year)
print(f"This person was born {years_ago} years ago")
print(f"(Current year: {current_year})")

**ANSWER:** This person was born **[VALUE]** years ago.

## Questions 5 & 6: Verify Birth Date Validity

Determine if the earliest birth date is correct using data analysis.

In [ ]:
# Get detailed information about oldest person
oldest_nconst = oldest_person.iloc[0]['nconst']
oldest_name = oldest_person.iloc[0]['primaryName']
oldest_death_year = oldest_person.iloc[0]['deathYear']
oldest_professions = oldest_person.iloc[0]['primaryProfession']

print(f"ANALYSIS FOR: {oldest_name} (ID: {oldest_nconst})")
print("=" * 60)
print(f"Birth year: {int(earliest_birth_year)}")
print(f"Death year: {oldest_death_year}")
print(f"Profession(s): {oldest_professions}")

# Check lifespan if death year exists
if pd.notna(oldest_death_year):
    oldest_death_year_int = int(oldest_death_year)
    lifespan = oldest_death_year_int - int(earliest_birth_year)
    print(f"\nCalculated lifespan: {lifespan} years")
    
    if lifespan > 120:
        print("⚠️  WARNING: Lifespan exceeds 120 years (biologically improbable)")
    elif lifespan < 0:
        print("⚠️  ERROR: Death year is before birth year (data error)")
    else:
        print("✓ Lifespan is within reasonable range")
else:
    print(f"\n⚠️  No death year recorded")
    print(f"If alive, this person would be {years_ago} years old")

# Check their work history
print("\n" + "=" * 60)
print("VERIFYING WORK HISTORY")
print("=" * 60)

title_principals = load_imdb_data('title.principals.tsv.gz')
person_works = title_principals[title_principals['nconst'] == oldest_nconst]

if len(person_works) > 0:
    # Merge with title_basics to get years
    person_works_with_years = person_works.merge(
        title_basics[['tconst', 'startYear', 'primaryTitle', 'titleType']], 
        on='tconst'
    )
    person_works_with_years['startYear'] = pd.to_numeric(
        person_works_with_years['startYear'], 
        errors='coerce'
    )
    person_works_with_years = person_works_with_years.dropna(subset=['startYear'])
    
    if len(person_works_with_years) > 0:
        earliest_work = person_works_with_years['startYear'].min()
        latest_work = person_works_with_years['startYear'].max()
        
        print(f"\nFirst work: {int(earliest_work)}")
        print(f"Last work: {int(latest_work)}")
        print(f"Career span: {int(latest_work - earliest_work)} years")
        
        # Calculate age at first work
        age_at_first_work = earliest_work - int(earliest_birth_year)
        print(f"\nAge at first work: {int(age_at_first_work)} years")
        
        if age_at_first_work < 5:
            print("⚠️  WARNING: Implausibly young to start working")
        elif age_at_first_work > 100:
            print("⚠️  WARNING: Implausibly old to still be working")
        else:
            print("✓ Age at first work seems reasonable")
        
        print(f"\nSample works ({len(person_works_with_years)} total):")
        display(person_works_with_years[['primaryTitle', 'startYear', 'category']].head(10))
else:
    print("No works found in database")

### ANSWER Questions 5 & 6: Birth Date Validity

**Is this birth date correct?** [YES/NO/UNCERTAIN - TO BE FILLED]

**Reasoning:**

Based on the analysis above, consider:

1. **Lifespan Analysis**: [Describe if lifespan is reasonable]
2. **Work History**: [Describe if age at first work makes sense]
3. **Career Duration**: [Describe if career span is plausible]
4. **Data Quality**: [Discuss possible data errors]

**Conclusion**: [Your detailed explanation here]

Common issues with very old birth dates:
- Data entry errors (wrong century)
- Default/placeholder values (e.g., 01/01/1800)
- Incomplete historical records
- Database migration artifacts

## Question 7: Most Recent Birth Date

In [ ]:
# Find most recent birth year
most_recent_birth_year = name_basics['birthYear'].max()
print(f"Most recent year of birth: {int(most_recent_birth_year)}")

# Find people with this birth year
youngest_people = name_basics[name_basics['birthYear'] == most_recent_birth_year]
print(f"\nNumber of people born in {int(most_recent_birth_year)}: {len(youngest_people)}")
print("\nExamples:")
display(youngest_people[['primaryName', 'birthYear', 'primaryProfession']].head(10))

**ANSWER:** The most recent year of birth is **[VALUE]**

## Question 8: Percentage Without Birth Date

In [ ]:
people_without_birth = name_basics['birthYear'].isna().sum()
total_people = len(name_basics)
percentage_without_birth = (people_without_birth / total_people) * 100

print(f"People without birth date: {people_without_birth:,}")
print(f"Total people: {total_people:,}")
print(f"\nPercentage without birth date: {percentage_without_birth:.2f}%")

**ANSWER:** **[VALUE]%** of people do not have a listed birth date.

## Question 9: Length of Longest "Short" After 1900

In [ ]:
# Convert columns to numeric
title_basics['startYear'] = pd.to_numeric(title_basics['startYear'], errors='coerce')
title_basics['runtimeMinutes'] = pd.to_numeric(title_basics['runtimeMinutes'], errors='coerce')

# Filter shorts after 1900
shorts_after_1900 = title_basics[
    (title_basics['titleType'] == 'short') & 
    (title_basics['startYear'] > 1900) &
    (title_basics['runtimeMinutes'].notna())
]

# Find longest short
longest_short_runtime = shorts_after_1900['runtimeMinutes'].max()
longest_short = shorts_after_1900[shorts_after_1900['runtimeMinutes'] == longest_short_runtime]

print(f"Length of longest short after 1900: {int(longest_short_runtime)} minutes")
print(f"\nDetails:")
display(longest_short[['primaryTitle', 'startYear', 'runtimeMinutes', 'genres']])

**ANSWER:** The longest "short" after 1900 is **[VALUE]** minutes long.

## Question 10: Length of Shortest "Movie" After 1900

In [ ]:
# Filter movies after 1900
movies_after_1900 = title_basics[
    (title_basics['titleType'] == 'movie') & 
    (title_basics['startYear'] > 1900) &
    (title_basics['runtimeMinutes'].notna()) &
    (title_basics['runtimeMinutes'] > 0)
]

# Find shortest movie
shortest_movie_runtime = movies_after_1900['runtimeMinutes'].min()
shortest_movie = movies_after_1900[movies_after_1900['runtimeMinutes'] == shortest_movie_runtime]

print(f"Length of shortest movie after 1900: {int(shortest_movie_runtime)} minutes")
print(f"\nDetails:")
display(shortest_movie[['primaryTitle', 'startYear', 'runtimeMinutes', 'genres']].head())

**ANSWER:** The shortest "movie" after 1900 is **[VALUE]** minutes long.

## Question 11: List of All Genres

In [ ]:
# Extract all unique genres
all_genres = title_basics['genres'].dropna()

genre_set = set()
for genres_string in all_genres:
    if genres_string and genres_string != '\\N':
        genre_list = genres_string.split(',')
        genre_set.update(genre_list)

unique_genres = sorted(list(genre_set))

print(f"Total number of unique genres: {len(unique_genres)}\n")
print("Complete list of genres:")
for i, genre in enumerate(unique_genres, 1):
    print(f"{i:2d}. {genre}")

# Calculate genre frequencies
genre_counts = {}
for genres_string in all_genres:
    if genres_string and genres_string != '\\N':
        for genre in genres_string.split(','):
            genre_counts[genre] = genre_counts.get(genre, 0) + 1

sorted_genres = sorted(genre_counts.items(), key=lambda x: x[1], reverse=True)
print("\n" + "=" * 60)
print("Top 10 most frequent genres:")
for genre, count in sorted_genres[:10]:
    print(f"{genre:20s}: {count:>8,} titles")

**ANSWER:** There are **[NUMBER]** unique genres in the dataset.

## Question 12: Highest Rated Comedy Movie

Find the comedy movie with the highest rating (ties broken by number of votes).

In [ ]:
# Merge title_basics with title_ratings
titles_with_ratings = title_basics.merge(title_ratings, on='tconst', how='inner')

# Filter comedy movies
comedy_movies = titles_with_ratings[
    (titles_with_ratings['titleType'] == 'movie') &
    (titles_with_ratings['genres'].notna()) &
    (titles_with_ratings['genres'].str.contains('Comedy', case=False, na=False))
]

print(f"Number of comedy movies with ratings: {len(comedy_movies):,}")

# Sort by rating (desc) then votes (desc)
comedy_movies_sorted = comedy_movies.sort_values(
    by=['averageRating', 'numVotes'], 
    ascending=[False, False]
)

# Get best movie
best_comedy = comedy_movies_sorted.iloc[0]

print("\n" + "=" * 60)
print("HIGHEST RATED COMEDY MOVIE")
print("=" * 60)
print(f"Title: {best_comedy['primaryTitle']}")
print(f"Year: {int(best_comedy['startYear']) if pd.notna(best_comedy['startYear']) else 'N/A'}")
print(f"Average Rating: {best_comedy['averageRating']}/10")
print(f"Number of Votes: {int(best_comedy['numVotes']):,}")
print(f"Genres: {best_comedy['genres']}")
print(f"IMDB ID: {best_comedy['tconst']}")

print("\n" + "=" * 60)
print("Top 10 Comedy Movies for Context:")
print("=" * 60)
display(comedy_movies_sorted[['primaryTitle', 'startYear', 'averageRating', 'numVotes']].head(10))

# Save for next question
best_comedy_tconst = best_comedy['tconst']

**ANSWER:** The highest rated comedy movie is **"[TITLE]"** with a rating of **[RATING]/10** and **[VOTES]** votes.

## Question 13: Director of the Movie

In [ ]:
# Find director(s) in title_crew
movie_crew = title_crew[title_crew['tconst'] == best_comedy_tconst]

print(f"Director(s) of '{best_comedy['primaryTitle']}':")
print("=" * 60)

if len(movie_crew) > 0:
    directors_ids = movie_crew.iloc[0]['directors']
    
    if pd.notna(directors_ids) and directors_ids != '\\N':
        director_id_list = directors_ids.split(',')
        
        for i, director_id in enumerate(director_id_list, 1):
            director_info = name_basics[name_basics['nconst'] == director_id]
            
            if len(director_info) > 0:
                director = director_info.iloc[0]
                print(f"\nDirector #{i}:")
                print(f"  Name: {director['primaryName']}")
                print(f"  ID: {director['nconst']}")
                print(f"  Birth Year: {int(director['birthYear']) if pd.notna(director['birthYear']) else 'N/A'}")
                print(f"  Professions: {director['primaryProfession']}")
                print(f"  Known For: {director['knownForTitles']}")
    else:
        print("No director information found")
else:
    print("No crew information found for this movie")

**ANSWER:** The director(s) of the movie is/are: **[NAME(S)]**

## Question 14: Alternate Titles for the Movie

In [ ]:
# Find alternate titles
alternate_titles = title_akas[title_akas['titleId'] == best_comedy_tconst]

print(f"Alternate titles for '{best_comedy['primaryTitle']}':")
print("=" * 60)
print(f"Total number of alternate titles: {len(alternate_titles)}\n")

if len(alternate_titles) > 0:
    # Display by region
    print("Sample alternate titles:")
    for idx, row in alternate_titles.head(20).iterrows():
        region = row['region'] if pd.notna(row['region']) else 'International'
        language = row['language'] if pd.notna(row['language']) else 'N/A'
        title = row['title']
        types = row['types'] if pd.notna(row['types']) else 'N/A'
        
        print(f"\nRegion: {region:10s} | Language: {language:10s}")
        print(f"Title: {title}")
        print(f"Type: {types}")
        print("-" * 60)
    
    # Summary by region
    print("\n" + "=" * 60)
    print("Summary by Region (Top 10):")
    print("=" * 60)
    region_counts = alternate_titles['region'].value_counts()
    print(region_counts.head(10))
else:
    print("No alternate titles found for this movie.")

**ANSWER:** The movie has **[NUMBER]** alternate titles in different regions/languages.

---
# Part 2: Kafka Stream Processing
---

## Overview

This section implements a **Kafka-based stream processing system** to monitor Wikipedia events in real-time for 5 selected entities from the IMDB dataset.

### Architecture:
1. **Producer**: Connects to Wikimedia EventStreams and publishes events to Kafka
2. **Kafka Broker**: Message queue (running locally via Docker)
3. **Consumer**: Processes events, calculates metrics, and triggers alerts
4. **Storage**: Saves metrics to JSON files (could be extended to database)

### Selected Entities:
We monitor 5 entities with active Wikipedia pages:

1. **The Shawshank Redemption** (Movie) - Classic film with frequent updates
2. **Christopher Nolan** (Director) - Active filmmaker with recent releases
3. **Leonardo DiCaprio** (Actor) - Highly visible celebrity
4. **Marvel Cinematic Universe** (Franchise) - Continuously updated franchise
5. **Academy Awards** (Event) - Annual event with significant activity

### Metrics Collected:
- Total number of edits per entity
- Number of unique editors
- Total bytes added/removed
- Edit type distribution (edit, new, log)
- Timestamp of last update

### Alert System:
- **Alert Type**: Major Edit Alert
- **Trigger**: When a single edit changes more than 5000 bytes
- **Action**: Event is written to separate alerts file
- **Use Case**: Detect significant content changes that may require review

### Files:
- `kafka_producer.py`: Publishes Wikipedia events to Kafka
- `kafka_consumer.py`: Processes events and calculates metrics
- `stream_data/metrics.json`: Collected metrics
- `stream_data/alerts.json`: Triggered alerts

### Running the System:
See the separate Python files and README for complete instructions.

---
# Conclusion
---

## Summary

This project successfully:
1. Analyzed **[X]** million IMDB records
2. Answered 14 specific data analysis questions
3. Implemented a Kafka-based stream processing system
4. Monitored real-time Wikipedia events
5. Created an alert system for significant changes

## Key Findings

[TO BE COMPLETED AFTER EXECUTION]

## Challenges Faced

[TO BE COMPLETED]

## Future Improvements

1. Store metrics in a proper database (PostgreSQL, MongoDB)
2. Add data visualization dashboard
3. Implement more sophisticated alert rules
4. Add machine learning for anomaly detection
5. Scale Kafka setup for production use